<img src="images/24.png" width="40%">

<img src="images/25.png" width="40%">

In [1]:
import torch

# -------------------------- 1.原始文本 --------------------------
raw_text = """the cat sit on mat
the dog run in park
the cat eat fish
the dog drink water
the cat sleep soft bed
the dog play ball
"""
print("===== 1.原始raw_text文本 =====")
print(raw_text)

# -------------------------- 2.构建词表映射表 map：字符 ↔ id --------------------------
# stoi : string‑to‑index  字符→数字id
# itos : index‑to‑string 数字id→字符
chars = sorted(list(set(raw_text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

print(f"\n===== 2.词表映射表 stoi (token‑id map)  vocab_size={vocab_size} =====")
for k, v in stoi.items():
    print(f"char:'{k}' → id:{v}")

# -------------------------- 3.编码、解码函数：文本 ↔ id序列 --------------------------
def encode(s: str):
    """文本字符串 → token id列表"""
    return [stoi[c] for c in s]

def decode(idx_list):
    """token id列表 → 还原字符串"""
    return "".join([itos[i] for i in idx_list])

# 把全部原始文本转为一长串token id张量
data = torch.tensor(encode(raw_text), dtype=torch.long)
print(f"\n===== 3.全部文本转为token id张量 =====")
print(f"data shape: {data.shape}")
print(f"data token id序列:\n{data.tolist()}")
print(f"还原回文本:\n{decode(data.tolist())}")

# -------------------------- 4.构造自回归语言模型训练样本（模仿nanoGPT get_batch） --------------------------
block_size = 8   # 上下文窗口长度：利用前面8个字符预测下一个字符
batch_size = 3   # 一批3条样本

max_start_pos = len(data) - block_size
print(f"\n===== 4.构造训练集参数 =====")
print(f"block_size(上下文窗口) = {block_size}")
print(f"max_start_pos 最大起始索引 = {max_start_pos}")

def get_batch(data, block_size, batch_size):
    """
    生成训练输入X、标签Y
    X：上文token id序列
    Y：向右错位一位，是X对应的目标下一个token
    """
    # 随机选取一批起始位置
    ix = torch.randint(low=0, high=max_start_pos, size=(batch_size,))
    X = torch.stack([data[i : i + block_size]      for i in ix])
    Y = torch.stack([data[i+1 : i+1 + block_size]  for i in ix])
    return X, Y, ix

X, Y, start_ix = get_batch(data, block_size, batch_size)

print("\n===== 5.get_batch得到训练样本 X(输入), Y(标签) =====")
print(f"start_ix 随机选取的样本起始位置: {start_ix.tolist()}")
print(f"X shape = {X.shape} , Y shape = {Y.shape}")

# 逐条打印：id序列 + 对应的可读文本
for b in range(batch_size):
    print(f"\n---样本 {b}---")
    print(f"X token id: {X[b].tolist()}")
    print(f"X文本: `{decode(X[b].tolist())}`")
    print(f"Y token id: {Y[b].tolist()}")
    print(f"Y文本: `{decode(Y[b].tolist())}`")

print("\n===== 关键映射关系说明 =====")
print("自回归任务：对于每一个位置t，用 X[t] 去预测 Y[t]")
print("即：上文序列 → 预测下一个token，Y相对于X整体向右错位1位。")


===== 1.原始raw_text文本 =====
the cat sit on mat
the dog run in park
the cat eat fish
the dog drink water
the cat sleep soft bed
the dog play ball


===== 2.词表映射表 stoi (token‑id map)  vocab_size=23 =====
char:'
' → id:0
char:' ' → id:1
char:'a' → id:2
char:'b' → id:3
char:'c' → id:4
char:'d' → id:5
char:'e' → id:6
char:'f' → id:7
char:'g' → id:8
char:'h' → id:9
char:'i' → id:10
char:'k' → id:11
char:'l' → id:12
char:'m' → id:13
char:'n' → id:14
char:'o' → id:15
char:'p' → id:16
char:'r' → id:17
char:'s' → id:18
char:'t' → id:19
char:'u' → id:20
char:'w' → id:21
char:'y' → id:22

===== 3.全部文本转为token id张量 =====
data shape: torch.Size([117])
data token id序列:
[19, 9, 6, 1, 4, 2, 19, 1, 18, 10, 19, 1, 15, 14, 1, 13, 2, 19, 0, 19, 9, 6, 1, 5, 15, 8, 1, 17, 20, 14, 1, 10, 14, 1, 16, 2, 17, 11, 0, 19, 9, 6, 1, 4, 2, 19, 1, 6, 2, 19, 1, 7, 10, 18, 9, 0, 19, 9, 6, 1, 5, 15, 8, 1, 5, 17, 10, 14, 11, 1, 21, 2, 19, 6, 17, 0, 19, 9, 6, 1, 4, 2, 19, 1, 18, 12, 6, 6, 16, 1, 18, 15, 7, 19, 1, 3, 6, 5, 0, 